# 05 Analysis queries (for charts and a dashboard)

Run each query and use the **+** button on the result to add a chart or send it to an AI/BI dashboard.

In [0]:
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA air_quality")

DataFrame[]

### 1. PM2.5 monthly trend

In [0]:
%sql
SELECT d.full_date AS month, f.concentration AS pm25
FROM gold_fact_air_quality f
JOIN gold_dim_date d      ON f.date_key = d.date_key
JOIN gold_dim_pollutant p ON f.pollutant_key = p.pollutant_key
WHERE p.pollutant_code = 'PM2.5'
ORDER BY d.full_date

month,pm25
2018-01-01,11.43
2018-02-01,16.75
2018-03-01,15.61
2018-04-01,16.34
2018-05-01,14.54
2018-06-01,17.28
2018-07-01,20.63
2018-08-01,25.84
2018-09-01,18.21
2018-10-01,14.5


### 2. Five worst months per pollutant

In [0]:
%sql
SELECT pollutant_code, year_month, concentration, worst_rank
FROM (
  SELECT p.pollutant_code, d.year_month, f.concentration,
         ROW_NUMBER() OVER (PARTITION BY p.pollutant_code ORDER BY f.concentration DESC) AS worst_rank
  FROM gold_fact_air_quality f
  JOIN gold_dim_date d      ON f.date_key = d.date_key
  JOIN gold_dim_pollutant p ON f.pollutant_key = p.pollutant_key
) ranked
WHERE worst_rank <= 5
ORDER BY pollutant_code, worst_rank

pollutant_code,year_month,concentration,worst_rank
CO,2019-09,0.888,1
CO,2017-07,0.716,2
CO,2019-08,0.708,3
CO,2017-11,0.699,4
CO,2017-09,0.682,5
NO2,2018-07,0.0096,1
NO2,2017-04,0.0087,2
NO2,2017-07,0.0082,3
NO2,2017-03,0.0082,4
NO2,2017-10,0.0082,5


### 3. April 2020 compared with the other Aprils (NO2)

In [0]:
%sql
SELECT
  ROUND(AVG(CASE WHEN d.year = 2020 THEN f.concentration END), 4)  AS april_2020,
  ROUND(AVG(CASE WHEN d.year <> 2020 THEN f.concentration END), 4) AS avg_other_aprils,
  ROUND(100.0 * (AVG(CASE WHEN d.year = 2020 THEN f.concentration END)
       / AVG(CASE WHEN d.year <> 2020 THEN f.concentration END) - 1), 1) AS pct_difference
FROM gold_fact_air_quality f
JOIN gold_dim_date d      ON f.date_key = d.date_key
JOIN gold_dim_pollutant p ON f.pollutant_key = p.pollutant_key
WHERE p.pollutant_code = 'NO2' AND d.month = 4

april_2020,avg_other_aprils,pct_difference
0.0032,0.0072,-55.6


### 4. PM2.5 three-month rolling average

In [0]:
%sql
SELECT d.full_date AS month,
       f.concentration AS pm25,
       ROUND(AVG(f.concentration) OVER (ORDER BY d.full_date ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 2) AS pm25_rolling_3m
FROM gold_fact_air_quality f
JOIN gold_dim_date d      ON f.date_key = d.date_key
JOIN gold_dim_pollutant p ON f.pollutant_key = p.pollutant_key
WHERE p.pollutant_code = 'PM2.5'
ORDER BY d.full_date

month,pm25,pm25_rolling_3m
2018-01-01,11.43,11.43
2018-02-01,16.75,14.09
2018-03-01,15.61,14.6
2018-04-01,16.34,16.23
2018-05-01,14.54,15.5
2018-06-01,17.28,16.05
2018-07-01,20.63,17.48
2018-08-01,25.84,21.25
2018-09-01,18.21,21.56
2018-10-01,14.5,19.52


Databricks visualization. Run in Databricks to view.